# B5 차원·군집 수·초기화 비교

태그 정제 개정 후 B4 최종 특징을 사용합니다. 이전 결과는 커밋 6e859f5에 남아 있습니다.
원본 공간의 silhouette를 고정 기준으로 사용하며 축소 공간 점수와 직접 혼합하지 않습니다.
후보·판단 절차를 아래에서 실행 전에 고정합니다. 독자 평점·test·cold_final은 사용하지 않습니다.


In [1]:
from pathlib import Path
from dataclasses import asdict
from time import perf_counter
import json
import platform
import numpy as np
import pandas as pd
import sklearn
from threadpoolctl import threadpool_limits
from sklearn.metrics import adjusted_rand_score
from bookrec.data import load_book_tags, load_books
from bookrec.models.content_based.tuning import ContentBasedConfig, make_content_based
from bookrec.models.content_based.clustering import (
    prepare_cluster_space, fit_book_clusters, cluster_diagnostics, cluster_overview,
    representative_books, load_selected_clusters,
)
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
OUT = ROOT / "results/content_based"
config = ContentBasedConfig(**json.loads((OUT / "B4_selected_config.json").read_text())["config"])
features = make_content_based(load_book_tags(), config).features
books = load_books()
DIMENSIONS = (None, 50, 100, 200)
K_CANDIDATES = (5, 8, 10, 12, 15, 20)
FIT_SEEDS = (42, 7, 2026)
SVD_SEED, MIN_BOOKS, MIN_GAIN, SAMPLE_SIZE = 42, 100, 0.0001, 2000
sample_ids = np.sort(np.random.default_rng(42).choice(features.book_ids, SAMPLE_SIZE, replace=False))
remaining_ids = np.setdiff1d(features.book_ids, sample_ids)
confirmation_ids = np.sort(np.random.default_rng(2026).choice(remaining_ids, SAMPLE_SIZE, replace=False))
assert not set(sample_ids) & set(confirmation_ids)
base_saved = json.loads((OUT / "B5_selected_clustering.json").read_text())
BASE_K = base_saved["k"]
print("특징", features.matrix.shape, "원본 기준 k", BASE_K)


특징 (10000, 33203) 원본 기준 k 10


## 1. 사전 고정한 비교와 채택 기준

원본·50·100·200차원 x k=5·8·10·12·15·20 x k-means 시드=42·7·2026의 72개 조합을 비교합니다.
SVD는 시드42, n_iter=7, k-means는 n_init=10/max_iter=300/tol=1e-4/Lloyd입니다.
모든 시드에서 최소 군집 100권·미배정 0권인 후보 중 고정 원본 공간 silhouette 3시드 평균이 최대인 조합을 고릅니다.
동점은 원본/낮은 차원, 작은 k 순서입니다. 배포용 학습 시드는 최고 시드를 고르지 않고 42로 고정합니다.

채택하려면 원본 기준선 대비 평균, 시드42, 겹치지 않는 확인 표본의 개선이 각각 0.0001보다 커야 합니다.
SVD를 선택하면 SVD 시드7·2026도 확인합니다. 그 두 결과와 원래 SVD시드42의 평균이 기준선보다
0.0001 넘게 높고 모든 결과가 크기·배정 조건을 만족해야 합니다. 이 값은 통계적 유의성 기준이 아닙니다.
별도 표본은 같은 카탈로그 안의 감도 확인이며 독립적인 일반화 평가가 아닙니다.


In [2]:
records, models, spaces, grid_labels = [], {}, {}, {}
def measure(stage, dimension, k, seed=42, svd_seed=42, ids=None, space=None):
    ids = sample_ids if ids is None else ids
    start = perf_counter()
    if space is None:
        space = prepare_cluster_space(features, dimension, seed=svd_seed)
    clusters = fit_book_clusters(features, k, seed=seed, space=space)
    per_book, _ = cluster_diagnostics(features, clusters, ids)
    sizes = np.bincount(clusters.labels[clusters.labels >= 0], minlength=k)
    row = {"candidate_id": f"{stage}_d{dimension or 0}_k{k}_fit{seed}_svd{svd_seed}", "stage": stage,
           "n_components": dimension or 0, "k": k, "fit_seed": seed, "svd_seed": svd_seed,
           "sample_seed": 2026 if stage == "confirmation" else 42,
           "sample_books": len(ids), "min_books": int(sizes.min()), "max_books": int(sizes.max()),
           "unassigned_books": int((clusters.labels < 0).sum()), "inertia": clusters.inertia,
           "n_iter": clusters.n_iter, "explained_variance_ratio": space.explained_variance_ratio,
           "eligible": bool(sizes.min() >= MIN_BOOKS and (clusters.labels >= 0).all()),
           "reference_silhouette": float(per_book.original_silhouette.mean()),
           "clustering_silhouette": float(per_book.clustering_silhouette.mean()),
           "reference_negative_rate": float((per_book.original_silhouette < 0).mean()),
           "seconds_fit_diagnostics": perf_counter() - start}
    records.append(row)
    pd.DataFrame(records).to_csv(OUT / "B5_improvement_comparison.csv", index=False)
    return clusters, per_book

with threadpool_limits(limits=2):
    for dimension in DIMENSIONS:
        spaces[dimension] = prepare_cluster_space(features, dimension, seed=SVD_SEED)
        for k in K_CANDIDATES:
            for seed in FIT_SEEDS:
                clusters, _ = measure("grid", dimension, k, seed=seed, space=spaces[dimension])
                grid_labels[(dimension, k, seed)] = clusters.labels
                if seed == 42:
                    models[(dimension, k)] = clusters
grid = pd.DataFrame(records)
for index, row in grid.iterrows():
    baseline_labels = models[(int(row.n_components) or None, row.k)].labels
    # NOTE: 군집 번호를 직접 맞추지 않고 같은 표현·k의 시드42 배정과 ARI로 비교한다.
    records[index]["ari_vs_seed42"] = float(adjusted_rand_score(
        baseline_labels, grid_labels[(int(row.n_components) or None, int(row.k), int(row.fit_seed))]))
summary = grid.groupby(["n_components", "k"]).agg(
    reference_mean=("reference_silhouette", "mean"),
    reference_std=("reference_silhouette", "std"),
    clustering_mean=("clustering_silhouette", "mean"),
    min_books=("min_books", "min"), eligible=("eligible", "all")).reset_index()
eligible = summary[summary.eligible].sort_values(
    ["reference_mean", "n_components", "k"], ascending=[False, True, True])
if eligible.empty:
    raise ValueError("군집 크기·배정 조건을 만족하는 후보가 없습니다.")
best = eligible.iloc[0]
dimension = int(best.n_components) or None
best_k = int(best.k)
proposed = models[(dimension, best_k)]
baseline = models[(None, BASE_K)]
baseline_mean = float(summary[(summary.n_components == 0) & (summary.k == BASE_K)].reference_mean.iloc[0])
print(summary.to_string(index=False))
print("후보", dimension, best_k, "원본 기준 시드 평균", baseline_mean)


 n_components  k  reference_mean  reference_std  clustering_mean  min_books  eligible
            0  5        0.051668       0.000324         0.051668       1103      True
            0  8        0.060360       0.000712         0.060360        392      True
            0 10        0.060802       0.000507         0.060802        387      True
            0 12        0.059662       0.003529         0.059662        379      True
            0 15        0.055692       0.001140         0.055692        291      True
            0 20        0.053013       0.000416         0.053013        197      True
           50  5        0.053582       0.000007         0.142205       1078      True
           50  8        0.054483       0.000042         0.154056        535      True
           50 10        0.060103       0.003320         0.172699        386      True
           50 12        0.062086       0.000103         0.183509        328      True
           50 15        0.058264       0.000947       

## 2. 확인 표본과 SVD 초기화 민감도

확인 표본은 선택 표본과 책 ID가 겹치지 않습니다. 확인 결과를 보고 다른 후보로 재탐색하지 않습니다.
SVD 초기화 비교에서는 k-means 시드를42로 고정해 두 종류의 난수를 구분합니다.


In [3]:
with threadpool_limits(limits=2):
    _, baseline_confirmation = measure("confirmation", None, BASE_K, ids=confirmation_ids, space=spaces[None])
    _, proposed_confirmation = measure("confirmation", dimension, best_k, ids=confirmation_ids, space=spaces[dimension])
    proposed_sample, _ = cluster_diagnostics(features, proposed, sample_ids)
    baseline_sample, _ = cluster_diagnostics(features, baseline, sample_ids)
    svd_checks = []
    if dimension is not None:
        for seed in (7, 2026):
            candidate, _ = measure("svd_stability", dimension, best_k, svd_seed=seed)
            svd_checks.append(records[-1])
confirmation_gain = float(proposed_confirmation.original_silhouette.mean() - baseline_confirmation.original_silhouette.mean())
seed42_gain = float(proposed_sample.original_silhouette.mean() - baseline_sample.original_silhouette.mean())
svd_mean = float(np.mean([proposed_sample.original_silhouette.mean(), *[r["reference_silhouette"] for r in svd_checks]]))
svd_ok = all(r["eligible"] for r in svd_checks) and svd_mean > baseline_sample.original_silhouette.mean() + MIN_GAIN
adopted = bool(best.reference_mean > baseline_mean + MIN_GAIN and seed42_gain > MIN_GAIN
               and confirmation_gain > MIN_GAIN and svd_ok)
final = proposed if adopted else baseline
final_dimension, final_k = (dimension, best_k) if adopted else (None, BASE_K)
print("채택:", adopted, "선택 표본 개선:", seed42_gain, "확인 표본 개선:", confirmation_gain, "SVD 시드 평균:", svd_mean)
# NOTE: 책별 silhouette 차이의 bootstrap은 같은 고정 군집 안의 표본 민감도이며 독립적 일반화 CI가 아니다.
delta = proposed_confirmation.original_silhouette.to_numpy() - baseline_confirmation.original_silhouette.to_numpy()
rng = np.random.default_rng(2026)
boot = np.array([rng.choice(delta, len(delta), replace=True).mean() for _ in range(2000)])
confirmation_interval = np.quantile(boot, [0.025, 0.975]).tolist()
print("확인 표본 차이의 조건부 95% bootstrap 구간:", confirmation_interval)


채택: False 선택 표본 개선: 0.0005978968831458278 확인 표본 개선: -0.000492950959734173 SVD 시드 평균: 0.06202909964644859
확인 표본 차이의 조건부 95% bootstrap 구간: [-0.0017909503906640269, 0.0008383950016397745]


## 3. 최종 결과와 다양한 대표 책

먼저 중심 거리만으로 고른 결과를 비교하고, 저자·명시적 시리즈 중복을 제한한 대표 책 5권을 저장합니다.
부족할 때만 시리즈·저자·제목 제한을 순서대로 완화하며 CSV의 candidate_rank·selection_rule에 기록합니다.
군집 태그는 원본 공간 평균, 대표 책의 거리는 실제 군집 좌표에서 계산합니다.


In [4]:
overview = cluster_overview(features, final)
plain = representative_books(features, final)
representatives = representative_books(features, final, books=books, diversify=True)
representatives = representatives.merge(books[["book_id", "title", "main_author"]], on="book_id", validate="many_to_one")
with threadpool_limits(limits=2):
    final_sample, diagnostic = cluster_diagnostics(features, final, sample_ids)
details = overview.rename(columns={"books": "cluster_books"}).merge(diagnostic.drop(columns="books"), on="cluster", validate="one_to_one")
representatives = representatives.merge(details, on="cluster", validate="many_to_one")
distance_change = float(representatives.distance.mean() - plain.distance.mean())
assignments = pd.DataFrame({"book_id": final.book_ids, "cluster": final.labels})
representatives.to_csv(OUT / "B5_improvement_representative_books.csv", index=False)
assignments.to_csv(OUT / "B5_improvement_assignments.csv", index=False)
settings = {
    "adopted": adopted, "feature_config": asdict(config), "n_components": final_dimension, "k": final_k,
    "svd_seed": 42, "svd_n_iter": 7, "fit_seed": 42, "n_init": 10, "max_iter": 300, "tol": 1e-4,
    "algorithm": "lloyd", "initialization": "k-means++", "metric": "euclidean",
    "sample_size": SAMPLE_SIZE, "selection_sample_seed": 42, "confirmation_sample_seed": 2026,
    "confirmation_disjoint": True, "min_cluster_books": MIN_BOOKS, "minimum_gain": MIN_GAIN,
    "criterion": "joint dimensions x k; 3-fit-seed fixed-original mean; size/coverage; mean/seed42/confirmation gains >0.0001; SVD-seed sensitivity gate",
    "reference_config": asdict(config), "reference_silhouette": float(final_sample.original_silhouette.mean()),
    "clustering_silhouette": float(final_sample.clustering_silhouette.mean()),
    "proposed_k": best_k, "proposed_n_components": dimension,
    "proposed_reference_mean": float(best.reference_mean), "baseline_reference_mean": baseline_mean,
    "proposed_confirmation_gain": confirmation_gain, "confirmation_conditional_ci95": confirmation_interval,
    "svd_seed_mean_reference_silhouette": svd_mean, "reference_space_changed": False,
    "explained_variance_ratio": final.space.explained_variance_ratio,
    "representative_policy": "author_series; relax series then author then title if insufficient",
    "representative_mean_distance_increase": distance_change,
    "python": platform.python_version(), "numpy": np.__version__, "sklearn": sklearn.__version__,
}
(OUT / "B5_improvement_selected_config.json").write_text(json.dumps(settings, ensure_ascii=False, indent=2) + "\n")
pd.DataFrame(records).to_csv(OUT / "B5_improvement_comparison.csv", index=False)
assert len(assignments) == assignments.book_id.nunique() == 10000
assert overview.books.sum() == 10000 and len(representatives) == final_k * 5
assert representatives.book_id.is_unique and representatives.title.notna().all()
for cluster, group in representatives.groupby("cluster"):
    assert assignments.set_index("book_id").cluster.loc[group.book_id].eq(cluster).all()
    assert group.distance.is_monotonic_increasing
    if group.selection_rule.eq("author_series").all():
        assert group.main_author.str.casefold().is_unique
print(json.dumps(settings, ensure_ascii=False, indent=2))
print(representatives[representatives["rank"] == 1].to_string(index=False))
print("다양성 제한에 따른 평균 거리 증가:", distance_change)


{
  "adopted": false,
  "feature_config": {
    "extended": false,
    "weighting": "binary",
    "min_df": 1,
    "sublinear_tf": true,
    "profile_weighting": "positive"
  },
  "n_components": null,
  "k": 10,
  "svd_seed": 42,
  "svd_n_iter": 7,
  "fit_seed": 42,
  "n_init": 10,
  "max_iter": 300,
  "tol": 0.0001,
  "algorithm": "lloyd",
  "initialization": "k-means++",
  "metric": "euclidean",
  "sample_size": 2000,
  "selection_sample_seed": 42,
  "confirmation_sample_seed": 2026,
  "confirmation_disjoint": true,
  "min_cluster_books": 100,
  "minimum_gain": 0.0001,
  "criterion": "joint dimensions x k; 3-fit-seed fixed-original mean; size/coverage; mean/seed42/confirmation gains >0.0001; SVD-seed sensitivity gate",
  "reference_config": {
    "extended": false,
    "weighting": "binary",
    "min_df": 1,
    "sublinear_tf": true,
    "profile_weighting": "positive"
  },
  "reference_silhouette": 0.06138707371977224,
  "clustering_silhouette": 0.06138707371977224,
  "proposed_k":

## 4. 공식 사용 경로 재현

최종 JSON으로 원본/SVD 여부·seed를 명시적으로 재현합니다. 없거나 잘못된 설정은 오류로 처리합니다.


In [5]:
with threadpool_limits(limits=2):
    replay_features, replay = load_selected_clusters()
np.testing.assert_array_equal(replay.book_ids, assignments.book_id)
np.testing.assert_array_equal(replay.labels, assignments.cluster)
replay_reps = representative_books(replay_features, replay, books=books, diversify=True)
np.testing.assert_array_equal(replay_reps.book_id, representatives.book_id)
np.testing.assert_allclose(replay_reps.distance, representatives.distance, atol=1e-7)
print("최종 로더: 10,000권 군집·대표 책 재현 통과")


최종 로더: 10,000권 군집·대표 책 재현 통과


## 5. 발표용 그림

원본 기준과 축소 공간을 구분하고, 오차 막대는 초기화 3시드의 표준편차로 표시합니다. 공용 save만 사용합니다.


In [6]:
import os
import tempfile
os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "bookrec-matplotlib"))
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from bookrec.plot_style import setup, save, MODEL_COLORS
setup()
fig, ax = plt.subplots()
dimension_view = summary[summary.k == BASE_K].sort_values("n_components")
x = np.arange(len(dimension_view))
ax.bar(x-0.18, dimension_view.reference_mean, width=0.36, color=MODEL_COLORS["cb"], label="Fixed original space")
ax.bar(x+0.18, dimension_view.clustering_mean, width=0.36, color=MODEL_COLORS["mf"], label="Clustering space")
ax.set_xticks(x, ["Original" if value == 0 else f"SVD {value}" for value in dimension_view.n_components])
ax.set_ylabel("Mean silhouette (3 k-means seeds)")
ax.set_title(f"B5: representation comparison at baseline k={BASE_K}")
ax.legend()
save(fig, "B5_improvement_dimension_comparison")
plt.close(fig)
fig, ax = plt.subplots()
for dimension_value, group in summary.groupby("n_components"):
    ax.errorbar(group.k, group.reference_mean, yerr=group.reference_std, marker="o", capsize=3,
                label="Original" if dimension_value == 0 else f"SVD {dimension_value}")
ax.set_xticks(K_CANDIDATES)
ax.set_xlabel("Number of clusters (k)")
ax.set_ylabel("Silhouette in fixed original space")
ax.set_title("B5: joint dimension and cluster-count comparison")
ax.legend(fontsize=9)
save(fig, "B5_improvement_k_stability")
plt.close(fig)
print("공용 save로 그래프 2개 저장")


공용 save로 그래프 2개 저장


## 사용과 한계

B4_selected_config.json은 추천기용, B5_selected_clustering.json은 원본 공간 비교용,
B5_improvement_selected_config.json은 현재 최종 군집용입니다. 과거 커밋과 현재 결과를 혼합하지 않습니다.
확인 표본도 군집 학습에 포함된 같은 카탈로그의 책이므로 독립 테스트가 아닙니다.
대표 책 다양성은 명시적 제목 패턴과 대표 저자 기준이며, 모든 시리즈·공저자 중복을 보장하지 않습니다.
